# Waymaker Artemis — Gemma 4 E4B CUDA 재현 노트북 (Mission 005-C)

무료 Colab T4에서 공식 `google/gemma-4-E4B-it` 체크포인트(고정 revision)를 NF4 4비트로 올리고,
2026-10-30 실행과 같은 세 가지 사례(근거 없음, 합성 근거 1건, 합성 근거 충돌)를 저장소 코드로 다시 돌립니다.

- **비용**: 무료 런타임만 씁니다. 런타임 유형은 `T4 GPU`로 고르고, 유료 GPU나 컴퓨팅 단위를 쓰지 않습니다.
- **다운로드**: 3번 셀만 모델을 받습니다(약 16GB). 이미 캐시에 있으면 받지 않습니다.
- **비밀값**: 토큰이 필요 없습니다. 공개 모델이라 Hugging Face 토큰 없이 받습니다. 노트북에 토큰을 적지 마세요.
- **해석 범위**: 실행과 응답 형식만 확인합니다. 합성 규칙은 실제 법령이 아니며, 법률 정확도와 Official Gold Accuracy는 `N/A`입니다.
- 이 노트북의 결과는 새로 실행한 결과로만 기록합니다. 이전 실행 기록을 재사용하지 않습니다.

In [ ]:
# 1. 환경 확인 (읽기 전용)
import subprocess, sys, shutil
print(sys.version)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('Disk free GB:', round(shutil.disk_usage('/content').free / 1e9, 2))

In [ ]:
# 2. 저장소(실험 폴더만)와 bitsandbytes 준비
# REF는 이 코드가 올라간 브랜치나 커밋입니다. 푸시 전이라면 클론이 실패합니다.
REF = 'feat/artemis-cuda-colab-inference'
!git clone --quiet --depth 1 --filter=blob:none --sparse --branch {REF} https://github.com/club-paradiso/Visable.git /content/Visable
!git -C /content/Visable sparse-checkout set experiments/waymaker-ai
!git -C /content/Visable rev-parse HEAD
%pip install -q "bitsandbytes>=0.45"

In [ ]:
# 3. 고정 revision 모델 받기 (이 셀만 네트워크로 가중치를 받습니다)
import json, shutil
from huggingface_hub import snapshot_download
policy = json.load(open('/content/Visable/experiments/waymaker-ai/configs/artemis_cuda_runtime.json'))
MODEL, REVISION = policy['model']['checkpoint'], policy['model']['revision']
assert (MODEL, REVISION) == ('google/gemma-4-E4B-it', 'ee0ef6023621cff504d758262d4e04895a5af4a2')
need = policy['limits']['min_free_disk_gb_for_download']
assert shutil.disk_usage('/content').free / 1e9 >= need, f'디스크 여유 공간이 {need}GB보다 적습니다'
snapshot = snapshot_download(MODEL, revision=REVISION, token=False)
print(snapshot)

In [ ]:
# 4. 사전 검사: 패키지 버전과 GPU만 봅니다. 모델은 아직 올리지 않습니다.
%cd /content/Visable/experiments/waymaker-ai
!python artemis_bootstrap/gemma4_cuda_runtime.py --probe

In [ ]:
# 5. 가중치 SHA-256 확인(16GB, 1~2분) 뒤 세 가지 합성 사례 실행. 오프라인으로만 읽고, 기록을 덮어쓰지 않습니다.
import datetime, os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = f'/content/artemis_cuda_record_{stamp}.json'
!python artemis_bootstrap/probe_gemma4_cuda.py --out {OUT} --max-new-tokens 96

In [ ]:
# 6. 결과 요약. 상태 0=completed, 2=generated_but_invalid, 1=failed, 3=not_executed
import json
record = json.load(open(OUT))
print('status:', record['execution_status'], '| reason:', record.get('not_executed_reason') or record.get('error'))
print('device:', record['environment'].get('cuda_device'), '| weights:', record['provenance'].get('weight_integrity'))
print('quantization verified:', record.get('load', {}).get('quantization', {}).get('passed'))
for case in record['cases']:
    print(case['case_id'], case['execution_status'], case.get('generated_token_count'), case.get('finish_reason'), repr(case.get('raw_output')))

In [ ]:
# 7. 기록 파일 내려받기. 저장소에 올릴 때는 이 파일을 그대로 runs/ 아래에 둡니다.
from google.colab import files
files.download(OUT)